# 文档证据：数字检索、表格结构与引用核验

状态：verified（仅人工结构化教学数据上的检索与引用检查）。没有运行 OCR、VLM、ColPali，也没有测量真实模型延迟。

先读[文档理解](../01-concepts/01-vision-and-document-ai.md)与[多模态检索](../01-concepts/03-multimodal-context-and-rag.md)，实现见[evidence.py](../05-code/evidence.py)。

In [1]:
DOMAIN="15-multimodal-and-embodied"
from pathlib import Path
import sys
# Find this topic whether Jupyter was started from the repository or lab directory.
ancestors = [Path.cwd(), *Path.cwd().parents]
repo = next(p for p in ancestors if (p / "10-Knowledge").is_dir())
sys.path.insert(0, str(repo / "10-Knowledge" / DOMAIN / "05-code"))
from evidence import CELLS, structured_answer, flattened_baseline, verify_claim
from copy import deepcopy

## 1. 原始教学表

latency 文档 v1，第 1 页，单位 ms。

| 模型 | mean | p95 |
|---|---:|---:|
| A | 40 | 100 |
| B | 60 | 120 |

新版本 v2 的 B/p95 改为 110 ms。数字 120 必须与 B 行、p95 列、单位和版本一起保存，单独搜索一个数没有意义。

In [2]:
for cell in CELLS:
    print(cell.evidence_id,cell.row,cell.column,cell.value,cell.unit,"page",cell.page,"bbox",cell.bbox)

v1-a-mean A mean 40 ms page 1 bbox (0.35, 0.3, 0.5, 0.38)
v1-a-p95 A p95 100 ms page 1 bbox (0.55, 0.3, 0.72, 0.38)
v1-b-mean B mean 60 ms page 1 bbox (0.35, 0.42, 0.5, 0.5)
v1-b-p95 B p95 120 ms page 1 bbox (0.55, 0.42, 0.72, 0.5)
v2-b-p95 B p95 110 ms page 1 bbox (0.55, 0.42, 0.72, 0.5)


## 2. 一个故意很弱的扁平基线

基线保留文档版本，却丢掉行列关系，只返回该版本第一个数字。这个对照不是生产检索器，作用是隔离“丢失结构”这一问题。三个问题的答案由表格直接给定。

In [3]:
queries=[dict(doc_id="latency",version=1,row="A",column="mean"),
         dict(doc_id="latency",version=1,row="B",column="p95"),
         dict(doc_id="latency",version=2,row="B",column="p95")]
gold=[40,120,110]
flat=[flattened_baseline(CELLS,**q) for q in queries]
structured=[structured_answer(CELLS,**q) for q in queries]
print("gold:",gold,"flat:",flat,"structured:",[a["value"] for a in structured])
flat_correct=sum(a==g for a,g in zip(flat,gold))
structured_correct=sum(a["value"]==g for a,g in zip(structured,gold))
assert flat_correct==2 and structured_correct==3
print({"flat_correct":flat_correct,"structured_correct":structured_correct,"n":3})

gold: [40, 120, 110] flat: [40, 40, 110] structured: [40, 120, 110]
{'flat_correct': 2, 'structured_correct': 3, 'n': 3}


## 3. 有引用 ID 不等于引用正确

对第二个问题，逐项检验错误数值、错误单位、错误页码和错误版本。我们只验证结构化数值关系，不处理任意自然语言的支持/矛盾关系。

In [4]:
answer=structured[1];query=queries[1]
assert verify_claim(CELLS,answer,query)
wrong_value={**answer,"value":60}
wrong_unit={**answer,"unit":"s"}
wrong_page=deepcopy(answer);wrong_page["citation"]["page"]=2
checks={"valid":verify_claim(CELLS,answer,query),
        "wrong_value":verify_claim(CELLS,wrong_value,query),
        "wrong_unit":verify_claim(CELLS,wrong_unit,query),
        "wrong_page":verify_claim(CELLS,wrong_page,query),
        "wrong_version":verify_claim(CELLS,answer,queries[2])}
print(checks)
assert checks=={"valid":True,"wrong_value":False,"wrong_unit":False,"wrong_page":False,"wrong_version":False}

{'valid': True, 'wrong_value': False, 'wrong_unit': False, 'wrong_page': False, 'wrong_version': False}


## 4. 找不到或找到多个匹配项时不猜答案
真实解析可能缺一行，也可能重复摄取。两种情况都不能随便取第一条当作确定证据。

In [5]:
for cells,query in [(CELLS,{**queries[0],"row":"C"}),(CELLS+CELLS,queries[0])]:
    try:
        structured_answer(cells,**query)
    except ValueError as error:
        print("refused:",error)
    else:
        raise AssertionError("expected ambiguity or missing evidence")

refused: missing or ambiguous evidence
refused: missing or ambiguous evidence


## 结论范围与下一步

在这 3 个人工问题中，结构查询给出 3 个预设正确答案，扁平基线给出 2 个。这个结果只证明结构字段能消除所构造的歧义，样本太小且人为设计，不能代表真实文档准确率。

要验证完整 Document AI，下一步需添加真实许可文档、OCR/表格解析、区域标注和未见文档测试集；分别测解析错误、召回错误、数值/单位错误和引用错误。练习：把 bbox 改成像素坐标但保留归一化检查，解释为何坐标体系必须成为 Schema 的一部分。

自检参考：在 1000×1000 图像中，`[550,420,720,500]` 与归一化的 `[.55,.42,.72,.50]` 可表示同一块区域；前者不能通过 `[0,1]` 检查。只除以宽高还不够：如果换成 PDF 左下角原点，必须转换 y 轴方向；本域 Cell 约定左上角原点、x 向右、y 向下。完整 Schema 还应保存坐标体系和原图尺寸。进阶的 [PDF 解析任务](../../../20-Projects/learning-workbench/README.md)输出的是文字起点，不是这些人工标注的矩形框。
